# Author Insights Notebook

This notebook provides insights about a specific author including:
- Author summary statistics
- Word cloud visualization
- Publication timeline by month

In [13]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from collections import Counter
import re

# Word cloud
from wordcloud import WordCloud

# Plotting
import seaborn as sns

# Set style (try different style names for compatibility)
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')
sns.set_palette("husl")

## Parameters

Configure the dataset path and author ID here.

Account Ids Parameters

In [14]:
account_ids = None  # single accountid or a list of accountids

Dataset Parameters

In [ ]:
# Dataset parameters
dataset_folder = "Labeled_Datasets/Ecuador"   # folder path
dataset_part = "Ecuador_part_3.gzip.parquet"
file_name = f"Topic_Ner_{dataset_part}"  # file name
dataset_procced_date = "2026_01_08"        # ("%Y_%m_%d")


# Aggregation mode: 'individual' or 'mean'
# - 'individual': Show statistics for each author separately
# - 'mean': Show aggregated/mean statistics across all authors
aggregation_mode = 'individual'  # Options: 'individual' or 'mean'


# Recommended: working directory is: "Backbone-Graph/src"

In [ ]:
# Column name variables
account_col = 'accountid'      # Author/account identifier column
post_time_col = 'post_time'    # Post timestamp column
topic_col = 'BERTopic_topic'            # Topic column
is_control_col = 'is_control'  # IO/Control classification column
post_language_col ='post_language'
text_col = 'post_text'

## Load Data

In [17]:
# Load the parquet file
file_name_no_suffixes = dataset_part.split(".")[0]
processed_dataset_path = Path("../../results") / dataset_folder / file_name_no_suffixes / dataset_procced_date
# Example: "../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10"

parquet_file_path = processed_dataset_path / file_name

print(f"Loading data from: {parquet_file_path}")

# Load the parquet file
df = pd.read_parquet(parquet_file_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Loading data from: ../../results/Labeled_Datasets/Ecuador/Ecuador_part_3/2026_01_08/Topic_Ner_Ecuador_part_3.gzip.parquet
Data loaded successfully!
Shape: (49925, 26)

Columns: ['postid', 'post_text', 'application_name', 'post_language', 'in_reply_to_postid', 'in_reply_to_accountid', 'post_time', 'accountid', 'account_profile_description', 'follower_count', 'following_count', 'account_creation_date', 'is_repost', 'reposted_accountid', 'reposted_postid', 'hashtags', 'urls', 'account_mentions', 'is_control', 'clean_text_with_entities', 'clean_text_without_enteties', 'translated_post', 'BERTopic_topic', 'BERTopic_prob', 'K_Means_topic', 'ner_entities']

First few rows:


,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,...,urls,account_mentions,is_control,clean_text_with_entities,clean_text_without_enteties,translated_post,BERTopic_topic,BERTopic_prob,K_Means_topic,ner_entities
0,43394c24c54037f763e8318026ffba9aeb9317f672,"Queda prohibido no sonreir a los problemas, no...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2013-02-21 16:02:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,...,[],[5ff80261b935621244fb38fff05231bf108ba8a488],False,"queda prohibido sonreir los problemas, luchar ...","queda prohibido sonreir los problemas, luchar ...","forbidden smile problems, fight want, abandon ...",2,1.000000,0,[]
1,1b6cb8bafe65e8b9e639b7e52075c8ebf34d854d6c,"Te encontraras con gente hipócrita, con amigos...",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,spa_Latn,None,None,2013-02-21 16:04:04,8949eab1b3f746ed66a069ae5d6c415a57dc546ffc,Agradecida con Dios y con la Vida. Ahora en #...,471,...,[],[],True,"te encontraras con gente hipócrita, con amigos...","te encontraras con gente hipócrita, con amigos...","meet hypocritical people, false friends, mourn...",2,0.885226,0,[]
2,4b69e40a8e23031f7ddc3832dfe5f289e851b56299,Aventarle un diccionario a alguien en la cabez...,62010408238d4a0f958ec97eec3db0a4f69bcd0d11,spa_Latn,None,None,2013-02-21 16:04:12,8949eab1b3f746ed66a069ae5d6c415a57dc546ffc,Agradecida con Dios y con la Vida. Ahora en #...,471,...,[],[],True,aventarle un diccionario alguien en la cabeza ...,aventarle un diccionario alguien en la cabeza ...,"put dictionary someone's head say, ""words hurt"".",-1,0.000000,0,[]
3,f810b69a925e539fb18b86ad0d227751cf072738bd,"Suena mi alarma del celu, tendre que levantarm...",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,spa_Latn,None,None,2013-02-21 16:04:16,8949eab1b3f746ed66a069ae5d6c415a57dc546ffc,Agradecida con Dios y con la Vida. Ahora en #...,471,...,[],[7e6b7dd1aa865b143fc37f4181e392613d1e6dfe3f],True,"suena mi alarma del celu, tendre que levantarm...","suena mi alarma del celu, tendre que levantarm...","cell phone alarm, fucking up.",61,0.722175,0,[]
4,94e00228f2ac99ef8ae06ab1c5419e74dfd7dc4699,En el mar la vida es mas sabrosa 8' @5b3c3e3a2...,e278f5ccf53957cccf9e97b2ae3fec72d5b892f850,spa_Latn,None,None,2013-02-21 16:15:00,4a08f9f3e1b73d62e982f899cdfb644e2101bd28a4,Grateful💕,587,...,[https://f87702c9a584719155693093bf96b50d218e8...,"[ad1f5918aa283b31785d589c2ade1c1ca176a23a47, 5...",False,en el mar la vida es mas sabrosa 8' mention_5b...,en el mar la vida es mas sabrosa 8' <user> <us...,sea life tastier 8' <user> <user> <user> <user...,-1,0.000000,3,[]


## Select Author(s)

If author_id is not set, you can select from the available authors. 
You can provide a single accountid or a list of accountids.

In [ ]:
# Normalize author_id to always be a list

# if author_id is not set, show top 10 authors by post count
if account_ids is None or len(account_ids) == 0:
    print("Author ID not set. Showing top 10 authors by post count:")
    top_authors = df.groupby(account_col).size().sort_values(ascending=False).head(10).reset_index(name="posts_count")
    display(top_authors)  
    print("\nPlease set author_id in the parameters cell above. \nAccount IDs (comma-separated).")


    account_ids = input("Account IDs (comma-separated): ")
    account_ids = [x.strip() for x in account_ids.split(",") if x.strip()]
    

# normalize author_id to always be a list

if isinstance(account_ids, (tuple, set)):
    account_ids = list(account_ids)
if isinstance(account_ids, str):
    account_ids = [account_ids]


print(f"account_ids = {account_ids}")

Author ID not set. Showing top 10 authors by post count:


,accountid,posts_count
0,71cc57155d061d447c2c65756c7dfd6873e403a097,6027
1,1b3972f155cfa2073bd97cfbced79a7baeb57289d8,3301
2,4a08f9f3e1b73d62e982f899cdfb644e2101bd28a4,1778
3,40a927417ec7dd26939b28d3e7eeb83eddd908f1c3,1535
4,22de67d991b31493165b9270f7d8b5b4ce712d5fc8,324
5,7660f778af4ab8f9d4f1356263df4d15cdb4110766,266
6,3bb1a651896e919384f4fd7c1a9464e09f8f41f6fa,238
7,2c6c0ef5f445760177b016e33b36809bde3c7ac241,230
8,86f64e2fce5d67c880f1d220f60205d26692ae9e6d,216
9,0974a37fdd7ca5cfd94bba466789d706c8d3c17edb,215



Please set author_id in the parameters cell above.
account_ids = ['71cc57155d061d447c2c65756c7dfd6873e403a097']


In [21]:
# Filter data for selected author(s)
filtered_df = df[df[account_col].isin(account_ids)].copy()
    
if len(filtered_df) == 0:
    raise ValueError(f"No posts found for the selected author(s) {account_ids}.")

found_authors = filtered_df[account_col].unique()
print(f"Found {len(filtered_df)} posts for {len(found_authors)} author(s)")
print(f"Author IDs: {list(found_authors)}")
if len(found_authors) < len(account_ids):
    missing = set(account_ids) - set(found_authors)
    print(f"Warning: {len(missing)} author ID(s) not found: {list(missing)}")

display(filtered_df.head())

Found 6027 posts for 1 author(s)
Author IDs: ['71cc57155d061d447c2c65756c7dfd6873e403a097']


,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,...,urls,account_mentions,is_control,clean_text_with_entities,clean_text_without_enteties,translated_post,BERTopic_topic,BERTopic_prob,K_Means_topic,ner_entities
0,43394c24c54037f763e8318026ffba9aeb9317f672,"Queda prohibido no sonreir a los problemas, no...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2013-02-21 16:02:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,...,[],[5ff80261b935621244fb38fff05231bf108ba8a488],False,"queda prohibido sonreir los problemas, luchar ...","queda prohibido sonreir los problemas, luchar ...","forbidden smile problems, fight want, abandon ...",2,1.00000,0,[]
25,9580df7eba66dd9738343ef6c75c1fd1e265b7e5da,#soybomberovoluntario qué pueden decir de esto...,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2013-02-21 17:28:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,...,[https://045e3d6032821c531e739d4a5f4ed11c220c0...,[],False,hashtag_soybomberovoluntario qué pueden decir ...,soybomberovoluntario qué pueden decir de esto?...,soybombervolunteer this? gye firefighters dema...,100,1.00000,2,[]
29,6b4508051198ea85470d0265c713bb6eee34e63bdf,"Visión 7: Rafael Correa reelecto, con la parti...",e3415c9cb59e8c2e081aab325f8e78eeef650d5b4b,spa_Latn,None,None,2013-02-21 17:31:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,...,[https://fd38af54b2c0636e7fb3087b7d06ca58aff09...,[fc997c4b0d9d56dd3d7ecb8fdba955c9e22d727a2a],False,"visión 7: rafael correa reelecto, con la parti...","visión 7: rafael correa reelecto, con la parti...","vision 7: re-elected rafael belt, participatio...",-1,0.00000,2,"[vision, rafael]"
33,a767b7e1873cf370860593cd3d747e751491826c01,. @670dbf5a15ea084c092e358c47493f2ba8d43caa53 ...,e3415c9cb59e8c2e081aab325f8e78eeef650d5b4b,spa_Latn,None,None,2013-02-21 17:51:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,...,[https://4a53ffc0725d81f1c17ee588c0e6edf39cc1d...,"[670dbf5a15ea084c092e358c47493f2ba8d43caa53, 2...",False,. mention_670dbf5a15ea084c092e358c47493f2ba8d4...,. <user> realiza jornada de adopción de perros...,. <user> organises day adoption dogs cats <url...,7,1.00000,3,[]
34,19601108ae701ccdc0a9d52303a8564fc4d66d5d20,Mi mejor psicóloga LA MÚSICA.,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2013-02-21 18:13:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,...,[],[0a9d2e2aee76a31724ae3fc0a3689cc53a54d63f20],False,mi mejor psicóloga la música.,mi mejor psicóloga la música.,best psychologist music.,22,0.98365,0,[]


## Author Summary Statistics

Create a dataframe with statistics for each author.

In [ ]:
# Function to calculate statistics for a single author
def calculate_author_stats(df : pd.DataFrame, account_ids : list[str] | str):
    """Calculate statistics for a single author's data."""
    
    if isinstance(account_ids, str):
        account_ids = [account_ids]

    author_stats_list = []
    
    for aid in account_ids:

        author_posts = df[df[account_col] == aid].copy()

        stats_dict = {}
        
        stats_dict[account_col] = aid
        stats_dict['total_posts'] = len(author_posts)
        
        # Date range
        author_posts[post_time_col] = pd.to_datetime(author_posts[post_time_col]) # convert to datetime
        stats_dict['first_post_date'] = author_posts[post_time_col].min()
        stats_dict['last_post_date'] = author_posts[post_time_col].max()
        # stats_dict['date_range_days'] = (stats_dict['last_post_date'] - stats_dict['first_post_date']).days
        stats_dict['months_active'] = author_posts[post_time_col].dt.to_period('M').nunique()
        stats_dict['avg_posts_per_month'] = stats_dict['total_posts'] / stats_dict['months_active'] if stats_dict['months_active'] > 0 else 0
        
        # Topics
        stats_dict['unique_topics'] = author_posts[topic_col].nunique()
        stats_dict['top_topic'] = author_posts[topic_col].value_counts().index[0] if len(author_posts) > 0 else None
        stats_dict['top_topic_count'] = author_posts[topic_col].value_counts().iloc[0] if len(author_posts) > 0 else 0

        
        # IO/Control classification
        stats_dict['io_rate'] = 1 - author_posts[is_control_col].mean()
        stats_dict['io_classification'] = 1 if stats_dict['io_rate'] >= 0.5 else 0

        # Language statistics
        stats_dict["top_lang"] = author_posts[post_language_col].value_counts().index[0]


        author_stats_list.append(stats_dict)

    
    all_stats_df = pd.DataFrame(author_stats_list)

    # all_stats_df['all_authors_average'] = all_stats_df.mean()

    return all_stats_df

In [25]:
calculate_author_stats(filtered_df, account_ids)

,accountid,total_posts,first_post_date,last_post_date,date_range_days,months_active,avg_posts_per_month,unique_topics,top_topic,top_topic_count,io_rate,io_classification,top_lang
0,71cc57155d061d447c2c65756c7dfd6873e403a097,6027,2013-02-21 16:02:00,2014-01-10 07:31:00,322,12,502.25,121,-1,3475,1.0,1,spa_Latn


## Word Cloud

Generate a word cloud from the author's posts.

# TODO: cheak until here

In [ ]:

for aid in account_ids:        


    if num_authors == 1:
        # Single author - show one word cloud
        author_id = unique_authors[0]
        author_text_df = filtered_df[filtered_df[account_col] == author_id]
        all_text = ' '.join(author_text_df[text_column].astype(str).apply(clean_text).dropna())
        
        if all_text.strip():
            wordcloud = WordCloud(
                width=800, 
                height=400, 
                background_color='white',
                max_words=100,
                colormap='viridis',
                relative_scaling=0.5,
                random_state=42,
                stopwords=STOPWORDS
            ).generate(all_text)
            
            plt.figure(figsize=(12, 6))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')
            plt.title(f'Word Cloud for Author {author_id}', fontsize=16, pad=20)
            plt.tight_layout()
            plt.show()
        else:
            print(f"No text content found for author {author_id}")
    
    else:
        # Multiple authors - show individual word clouds + combined
        # Calculate grid size for individual word clouds
        cols = min(3, num_authors)
        rows = (num_authors + cols - 1) // cols
        
        # Create figure for individual word clouds
        fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
        if num_authors == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        # Generate word cloud for each author
        for idx, aid in enumerate(unique_authors):
            author_text_df = filtered_df[filtered_df[account_col] == aid]
            all_text = ' '.join(author_text_df[text_column].astype(str).apply(clean_text).dropna())
            
            if all_text.strip():
                wordcloud = WordCloud(
                    width=800, 
                    height=400, 
                    background_color='white',
                    max_words=100,
                    colormap='viridis',
                    relative_scaling=0.5,
                    random_state=42,
                    stopwords=STOPWORDS
                ).generate(all_text)
                
                axes[idx].imshow(wordcloud, interpolation='bilinear')
                axes[idx].axis('off')
                axes[idx].set_title(f'Author {aid}', fontsize=12, pad=10)
            else:
                axes[idx].text(0.5, 0.5, f'No text content\nfor author {aid}', 
                              ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('off')
                axes[idx].set_title(f'Author {aid}', fontsize=12, pad=10)
        
        # Hide unused subplots
        for idx in range(num_authors, len(axes)):
            axes[idx].axis('off')
        
        plt.suptitle('Word Clouds by Author', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        
        # Create combined word cloud for all authors
        print("\n" + "="*80)
        print("Combined Word Cloud (All Authors)")
        print("="*80 + "\n")
        
        # Combine all text from all authors
        all_authors_text = ' '.join(filtered_df[text_column].astype(str).apply(clean_text).dropna())
        
        if all_authors_text.strip():
            combined_wordcloud = WordCloud(
                width=1200, 
                height=600, 
                background_color='white',
                max_words=150,
                colormap='plasma',
                relative_scaling=0.5,
                random_state=42,
                stopwords=STOPWORDS
            ).generate(all_authors_text)
            
            plt.figure(figsize=(14, 7))
            plt.imshow(combined_wordcloud, interpolation='bilinear')
            plt.axis('off')
            plt.title(f'Combined Word Cloud - All {num_authors} Authors', fontsize=18, fontweight='bold', pad=20)
            plt.tight_layout()
            plt.show()
        else:
            print("No text content found for combined word cloud.")
else:
    print(f"Text column '{text_column}' not found in the dataframe.")


## Publication Timeline by Month

Visualize the number of posts published by the author each month.

In [ ]:
# Create publication timeline by month
if post_time_col in filtered_df.columns:
    # Ensure post_time is datetime
    filtered_df[post_time_col] = pd.to_datetime(filtered_df[post_time_col])
    filtered_df['year_month'] = filtered_df[post_time_col].dt.to_period('M')
    
    if aggregation_mode == 'individual' and len(filtered_df[account_col].unique()) > 1:
        # Create separate timeline for each author
        num_authors = len(filtered_df[account_col].unique())
        cols = min(2, num_authors)
        rows = (num_authors + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 5*rows))
        if num_authors == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, aid in enumerate(filtered_df[account_col].unique()):
            author_monthly_df = filtered_df[filtered_df[account_col] == aid]
            monthly_counts = author_monthly_df.groupby('year_month').size().reset_index(name='post_count')
            monthly_counts['year_month_str'] = monthly_counts['year_month'].astype(str)
            
            bars = axes[idx].bar(range(len(monthly_counts)), monthly_counts['post_count'], 
                               color='steelblue', alpha=0.7, edgecolor='navy', linewidth=1.5)
            
            axes[idx].set_xlabel('Month', fontsize=10, fontweight='bold')
            axes[idx].set_ylabel('Number of Posts', fontsize=10, fontweight='bold')
            axes[idx].set_title(f'Author {aid}', fontsize=12, fontweight='bold')
            axes[idx].set_xticks(range(len(monthly_counts)))
            axes[idx].set_xticklabels(monthly_counts['year_month_str'], rotation=45, ha='right', fontsize=8)
            axes[idx].grid(axis='y', alpha=0.3, linestyle='--')
            
            # Add value labels
            for bar, count in zip(bars, monthly_counts['post_count']):
                height = bar.get_height()
                axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                             f'{int(count)}', ha='center', va='bottom', fontsize=7)
        
        # Hide unused subplots
        for idx in range(num_authors, len(axes)):
            axes[idx].axis('off')
        
        plt.suptitle('Publication Timeline by Author', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        # Aggregated timeline (sum across all authors)
        monthly_counts = filtered_df.groupby('year_month').size().reset_index(name='post_count')
        monthly_counts['year_month_str'] = monthly_counts['year_month'].astype(str)
        
        # Create the plot
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Bar plot
        bars = ax.bar(range(len(monthly_counts)), monthly_counts['post_count'], 
                     color='steelblue', alpha=0.7, edgecolor='navy', linewidth=1.5)
        
        # Customize
        ax.set_xlabel('Month', fontsize=12, fontweight='bold')
        ax.set_ylabel('Number of Posts', fontsize=12, fontweight='bold')
        title = f'Publication Timeline - {len(filtered_df[account_col].unique())} Author(s)'
        if aggregation_mode == 'mean':
            title += ' (Aggregated)'
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
        ax.set_xticks(range(len(monthly_counts)))
        ax.set_xticklabels(monthly_counts['year_month_str'], rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Add value labels on bars
        for bar, count in zip(bars, monthly_counts['post_count']):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(count)}',
                   ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print(f"\nPublication Statistics:")
        print(f"  Total months with posts: {len(monthly_counts)}")
        print(f"  Average posts per month: {monthly_counts['post_count'].mean():.1f}")
        print(f"  Maximum posts in a month: {monthly_counts['post_count'].max()} ({monthly_counts.loc[monthly_counts['post_count'].idxmax(), 'year_month_str']})")
        print(f"  Minimum posts in a month: {monthly_counts['post_count'].min()} ({monthly_counts.loc[monthly_counts['post_count'].idxmin(), 'year_month_str']})")
        
else:
    print(f"'{post_time_col}' column not found in the dataframe. Cannot create timeline.")

## Summary DataFrame

The summary dataframe contains all statistics for each author. You can use this dataframe for further analysis.

In [ ]:
# Display the summary dataframe for easy access
print("Summary DataFrame (available as 'summary_df' variable):")
print("=" * 80)
display(summary_df)

# You can also export it if needed
# summary_df.to_csv('author_summary.csv', index=False)
# summary_df.to_parquet('author_summary.parquet', index=False)

## Additional Visualizations

Optional: Additional insights and visualizations.

In [ ]:
# Additional visualizations (optional)
# Create a figure with multiple subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Topic distribution (if available)
if topic_col in df.columns:
    topic_counts = df[topic_col].value_counts().head(10)
    axes[0].barh(range(len(topic_counts)), topic_counts.values, color='coral', alpha=0.7)
    axes[0].set_yticks(range(len(topic_counts)))
    axes[0].set_yticklabels([f'Topic {t}' for t in topic_counts.index])
    axes[0].set_xlabel('Number of Posts', fontweight='bold')
    axes[0].set_title('Top 10 Topics by Post Count', fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    axes[0].invert_yaxis()
else:
    axes[0].text(0.5, 0.5, 'Topic data not available', 
                ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Topic Distribution', fontweight='bold')

# 2. Posts per day of week (if post_time available)
if post_time_col in df.columns:
    df['day_of_week'] = df[post_time_col].dt.day_name()
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_counts = df['day_of_week'].value_counts().reindex(day_order, fill_value=0)
    
    axes[1].bar(range(len(day_counts)), day_counts.values, color='mediumseagreen', alpha=0.7)
    axes[1].set_xticks(range(len(day_counts)))
    axes[1].set_xticklabels([d[:3] for d in day_counts.index], rotation=0)
    axes[1].set_ylabel('Number of Posts', fontweight='bold')
    axes[1].set_title('Posts by Day of Week', fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Post time data not available', 
                ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Posts by Day of Week', fontweight='bold')

title = f'Additional Insights - {len(df[account_col].unique())} Author(s)'
if aggregation_mode == 'mean' and len(df[account_col].unique()) > 1:
    title += ' (Aggregated)'
plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()